Libraries


In [ ]:
import pandas as pd
import numpy as np
import string
import joblib
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
from nltk.stem import WordNetLemmatizer
from openpyxl import load_workbook


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


load trained model

In [ ]:
model = joblib.load('/content/drive/MyDrive/Xetalabs/Model/NRM_Desc_pred.pkl')

Get the file path

In [ ]:
file_path=input()

/content/drive/MyDrive/Xetalabs/Test/Testing_data.xlsx


read file

In [ ]:
data1=pd.read_excel(file_path)

In [ ]:
x_test1=data1['Description']

remove punctuation

In [ ]:
def remove_punctuation(text):
    translator = str.maketrans(" ", " ", string.punctuation)
    return text.translate(translator)

In [ ]:
x_test=[]
for text in x_test1:
  text_without_punctuation = remove_punctuation(str(text))
  x_test.append(text_without_punctuation.lower())

In [ ]:
remove numbers

In [ ]:
def remove_numbers(text):
    text_without_numbers = re.sub(r'\d+', '', text)
    return text_without_numbers

In [ ]:
x1=[]
for text in x_test:
  without_num=remove_numbers(text)
  x1.append(without_num)

Lemmetization

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
stop_words=set(stopwords.words("english"))
def remove_stopwords(text):
    # Tokenize the text into individual words
    tokens = word_tokenize(text)

    # Remove stopwords from the tokens
    filtered_tokens = [lemmatizer.lemmatize(word) for word in tokens if word.lower() not in stop_words]

    # Join the filtered tokens back into a single text string
    text_without_stopwords = ' '.join(filtered_tokens)

    return text_without_stopwords

Remove stop words

In [ ]:
x_test1=[]
for text in x1:
  without_stopwords=remove_stopwords(text)
  x_test1.append(without_stopwords)

load vectorizer

In [ ]:
tfidf=joblib.load('/content/drive/MyDrive/Xetalabs/Model/desc_vectorizer.pkl')#load vectorizer

In [ ]:
x_test1=tfidf.transform(x_test1)#transform data to vectorized form

In [ ]:
predicted_test=model.predict(x_test1)
proba=model.predict_proba(x_test1)

print output into the same excel file

In [ ]:
workbook = load_workbook(file_path)# excel path

worksheet = workbook.active

column_letter1 = 'B'
column_letter2 = 'C'
column_letter3 = 'D'
worksheet[f'{column_letter1}{1}'] = "Predicted NRM Code"
worksheet[f'{column_letter2}{1}'] = "Probability Value"
worksheet[f'{column_letter3}{1}'] = "Ture/False"
for i, value in enumerate(predicted_test, start=2):
    worksheet[f'{column_letter1}{i}'] = value

for index,i in enumerate(proba,start=2):
    worksheet[f'{column_letter2}{index}'] = np.max(i)
    worksheet[f'{column_letter3}{index}'] = 'True' if np.max(i) == 1 else 'False'
workbook.save(file_path)